# Import Library

In [1]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, Callback

2025-05-05 10:36:13.267532: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746416173.357782  202741 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746416173.395557  202741 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746416173.530817  202741 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746416173.530892  202741 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746416173.530894  202741 computation_placer.cc:177] computation placer alr

In [2]:
data = pd.read_csv('../data/clean_review_70k.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   content        70000 non-null  object
 1   clean_review   70000 non-null  object
 2   label_lexicon  70000 non-null  object
dtypes: object(3)
memory usage: 1.6+ MB


In [3]:
# Split data
X = data['clean_review'].values
y = data['label_lexicon'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [4]:
vocab_size = 10000
max_length = 50
trunc_type='post'
padding_type='post'
oov_tok = "<OOV>"

In [5]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding=padding_type)
X_val_pad = pad_sequences(X_val_seq, maxlen=max_length, padding=padding_type)

In [6]:
# One-hot encoding langsung dari string label
y_train_enc = pd.get_dummies(y_train).values
y_val_enc = pd.get_dummies(y_val).values

# Cek bentuk dan label
print("Shape y_train_enc:", y_train_enc.shape)
print("Columns:", pd.get_dummies(y_train).columns.tolist())

Shape y_train_enc: (56000, 3)
Columns: ['negative', 'neutral', 'positive']


# Model Deep Learning

In [7]:
# Callback kustom
class StopTrainingOnValAccuracy(Callback):
    def __init__(self, target_accuracy):
        super(StopTrainingOnValAccuracy, self).__init__()
        self.target_accuracy = target_accuracy

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get("val_accuracy")
        if val_acc is not None and val_acc >= self.target_accuracy:
            print(f"\n🔔 Val accuracy {val_acc:.4f} mencapai target {self.target_accuracy}. Menghentikan training.")
            self.model.stop_training = True

In [8]:
model_simpleRNN = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length),
    layers.Dropout(0.5),
    layers.SimpleRNN(64, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dense(24, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(12, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.Dense(8, activation='relu'),
    layers.Dense(6, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model_simpleRNN.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_simpleRNN.summary()

/home/atalamahardika/tf219/venv_gpu/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1746416183.520125  202741 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3586 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
history_simpleRNN = model_simpleRNN.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_pad, y_val_enc),
    callbacks=[EarlyStopping(monitor='val_loss', patience=3),
               StopTrainingOnValAccuracy(target_accuracy=0.95)]
)

Epoch 1/50


I0000 00:00:1746416186.532985  202876 service.cc:152] XLA service 0x7f1ac0012100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1746416186.533067  202876 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2025-05-05 10:36:26.595299: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1746416187.002701  202876 cuda_dnn.cc:529] Loaded cuDNN version 90300


   7/1750 ━━━━━━━━━━━━━━━━━━━━ 35s 21ms/step - accuracy: 0.3430 - loss: 1.1674

I0000 00:00:1746416191.474525  202876 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1750/1750 ━━━━━━━━━━━━━━━━━━━━ 37s 17ms/step - accuracy: 0.6577 - loss: 0.8396 - val_accuracy: 0.8154 - val_loss: 0.4544
Epoch 2/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - accuracy: 0.8099 - loss: 0.4680 - val_accuracy: 0.8568 - val_loss: 0.3974
Epoch 3/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - accuracy: 0.8595 - loss: 0.3793 - val_accuracy: 0.8671 - val_loss: 0.3660
Epoch 4/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - accuracy: 0.8786 - loss: 0.3312 - val_accuracy: 0.8852 - val_loss: 0.3200
Epoch 5/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 27s 16ms/step - accuracy: 0.8921 - loss: 0.2881 - val_accuracy: 0.8861 - val_loss: 0.3101
Epoch 6/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9045 - loss: 0.2542 - val_accuracy: 0.8856 - val_loss: 0.3081
Epoch 7/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 27s 16ms/step - accuracy: 0.9118 - loss: 0.2346 - val_accuracy: 0.8946 - val_loss: 0.2907
Epoch 8/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 27s 16ms/step - accuracy: 0.9149 - loss: 0.22

In [10]:
model_gru = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length),
    layers.Dropout(0.5),
    layers.GRU(64, recurrent_dropout=0.5, kernel_regularizer=tf.keras.regularizers.l2(0.001), return_sequences=True),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    layers.Dense(24, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(12, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.Dense(8, activation='relu'),
    layers.Dense(6, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model_gru.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_gru.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
history_gru = model_gru.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_pad, y_val_enc),
    callbacks=[EarlyStopping(monitor='val_loss', patience=3),
               StopTrainingOnValAccuracy(target_accuracy=0.95)]
)

Epoch 1/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 332s 187ms/step - accuracy: 0.6696 - loss: 0.7762 - val_accuracy: 0.8199 - val_loss: 0.4447
Epoch 2/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 324s 185ms/step - accuracy: 0.8285 - loss: 0.4348 - val_accuracy: 0.8563 - val_loss: 0.3993
Epoch 3/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 317s 181ms/step - accuracy: 0.8485 - loss: 0.3916 - val_accuracy: 0.8193 - val_loss: 0.4337
Epoch 4/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 324s 185ms/step - accuracy: 0.8661 - loss: 0.3610 - val_accuracy: 0.8473 - val_loss: 0.3899
Epoch 5/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 318s 182ms/step - accuracy: 0.8675 - loss: 0.3492 - val_accuracy: 0.8600 - val_loss: 0.3641
Epoch 6/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 321s 183ms/step - accuracy: 0.8816 - loss: 0.3150 - val_accuracy: 0.8795 - val_loss: 0.3245
Epoch 7/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 320s 183ms/step - accuracy: 0.8904 - loss: 0.2906 - val_accuracy: 0.8836 - val_loss: 0.3080
Epoch 8/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 332s 190ms/step - ac

In [12]:
model_custom = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model_custom.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_custom.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
history_custom = model_custom.fit(
    X_train_pad, y_train_enc,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_pad, y_val_enc),
    callbacks=[EarlyStopping(monitor='val_loss', patience=3),
               StopTrainingOnValAccuracy(target_accuracy=0.95)]
)

Epoch 1/50


2025-05-05 11:53:59.323845: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_560', 12 bytes spill stores, 12 bytes spill loads

2025-05-05 11:53:59.652539: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_560', 4 bytes spill stores, 4 bytes spill loads



1750/1750 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.6846 - loss: 0.7544 - val_accuracy: 0.8381 - val_loss: 0.4302
Epoch 2/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8355 - loss: 0.4247 - val_accuracy: 0.8599 - val_loss: 0.3876
Epoch 3/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8542 - loss: 0.3826 - val_accuracy: 0.8670 - val_loss: 0.3688
Epoch 4/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8697 - loss: 0.3515 - val_accuracy: 0.8628 - val_loss: 0.3625
Epoch 5/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8810 - loss: 0.3222 - val_accuracy: 0.8698 - val_loss: 0.3491
Epoch 6/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8874 - loss: 0.3063 - val_accuracy: 0.8759 - val_loss: 0.3380
Epoch 7/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8913 - loss: 0.2899 - val_accuracy: 0.8676 - val_loss: 0.3432
Epoch 8/50
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8957 - loss: 0.2803 - val_accura

## Kesimpulan

Dari hasil 3 percobaan diatas menggunakan algoritma simpleRNN, GRU, dan custom berikut ringkasannya :  
Model       | Training Accuracy | Training Loss | Validation Accuracy   | Validation Loss
------------|-------------------|---------------|-----------------------|----------------
SimpleRNN   | 0.9395            | 0.1650        | 0.9054                | 0.2890
GRU         | 0.9287            | 0.1917        | 0.9064                | 0.2760    
Custom      | 0.9023            | 0.2599        | 0.8701                | 0.3441

Model terbaik untuk inference adalah GRU karena menunjukkan `val_accuracy` yang tinggi dan `val_loss` yang rendah menandakan bahwa model ini mampu menggeneralisasi dengan baik.

# Inference

In [27]:
text = ["game pretty fun easy learn high graphic quality matchmaking put balanced player"]

In [15]:
def pipeline_inference(text):
  inference_seq = tokenizer.texts_to_sequences(text)
  inference_pad = pad_sequences(inference_seq, padding=padding_type, maxlen=max_length)
  return inference_pad

In [16]:
pipeline_inference(text)

array([[  2, 262,  37, 288, 656,  89,  80, 446,  19, 207, 343,   3,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0]],
      dtype=int32)

In [28]:
result = model_gru.predict(pipeline_inference(text))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step


In [29]:
print("Hasil prediksi model GRU:")
print(result)

predicted_labels = result.argmax(axis=1)
print("Label prediksi model GRU:")
print(predicted_labels)

label_map = {0: "Negatif", 1: "Netral", 2: "Positif"}
print("Prediksi Sentimen:", label_map[predicted_labels[0]])


Hasil prediksi model GRU:
[[9.3701369e-10 8.5175270e-06 9.9999154e-01]]
Label prediksi model GRU:
[2]
Prediksi Sentimen: Positif
